The notebook mirros the train_from_scratch.py which is main about llama3 structure and focus on MLA part integration.

Note that "naive" is like MHA mode and "absorb" is MQA mode

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import math
import random
from typing import List, Optional, Tuple, Union, Literal
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn
import os
from torch.utils.data import IterableDataset, Dataset
import json
import numpy as np
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers import PretrainedConfig
from transformers import (Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer,
                          DefaultDataCollator, DataCollatorForTokenClassification, AutoConfig,
                          TextStreamer)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## Pre-Training

In [23]:
# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        # rsqrt = 1/sqrt(x)
        result = self.weight * (hidden_states * torch.rsqrt(torch.mean(hidden_states * hidden_states, dim=-1, keepdim=True) + self.variance_epsilon))
        return result

In [24]:
# RoPE
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotate_pos_emb(q, k, cos, sin, unsqueeze_dim=2):
    
    cos = cos.unsqueeze(unsqueeze_dim) # (1, seq_len, 1, dim)
    sin = sin.unsqueeze(unsqueeze_dim) # (1, seq_len, 1, dim)
   
    q_embed = (q*cos) + (rotate_half(q)*sin)  # (batch_size, seq_len, head_num, dim) * (1, seq_len, 1, dim) = (batch_size, seq_len, head_num, dim) 广播
    k_embed = (k*cos) + (rotate_half(k)*sin)  # (batch_size, seq_len, head_num, dim) * (1, seq_len, 1, dim) = = (batch_size, seq_len, head_num, dim) 广播
    
    return q_embed, k_embed

class RotaryEmbedding(nn.Module):
    # Here is slight different than the slides, originally is [x1, x2, x3, x4,. ...] * [cos(m * theta_1), cos(m * theta_1), cos(m * theta_2), cos(m * theta_2), ..cos(m * theta_d/2)] 
    # + [-x2, x1, -x4, x3, ...] * [sin(m * theta_1), sin(m * theta_1), sin(m * theta_2), sin(m * theta_2), ..sin(m * theta_d/2)], which is equivalently to be 
    # [x1, x3, ..., x2, x4,. ...] * [cos(m * theta_1), cos(m * theta_2), ..., cos(m * theta_1), cos(m * theta_2), ..cos(m * theta_d/2)] 
    # + [-x2, -x4,..., x1, x3, ...] * [sin(m * theta_1), sin(m * theta_2), ..., sin(m * theta_1), sin(m * theta_2), ..sin(m * theta_d/2)]
    def __init__(self, dim, max_seq_len=2048):
        super(RotaryEmbedding, self).__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))  # (dim/2)
        t = torch.arange(max_seq_len).float().unsqueeze(1)  # (max_seq_len, 1)
        freqs = t @ inv_freq.unsqueeze(0)  #(max_seq_len, 1)*(1, dim/2) = (max_seq_len, dim/2), e.g. m * theta_i part in the slides
        freqs = torch.cat((freqs, freqs), dim=-1)  # (max_seq_len, dim)
        
        self.register_buffer("cos_cached", freqs.cos())
        self.register_buffer("sin_cached", freqs.sin())
        
    def forward(self, q, k, start_pos=0):
        # During decode with KV cache, the new token sits at absolute position `start_pos`,
        # so slice cos/sin from there, not from 0. With start_pos=0 this matches old behavior.
        seq_len = q.shape[1]
        cos = self.cos_cached[start_pos:start_pos+seq_len, :].unsqueeze(0)  # (1, seq_len, dim)
        sin = self.sin_cached[start_pos:start_pos+seq_len, :].unsqueeze(0)  # (1, seq_len, dim)
        return apply_rotate_pos_emb(q, k, cos, sin)
    

In [25]:
world_size = 1
rank = 0
block_size = 128
gemm_impl: Literal["bf16", "fp8"] = "bf16"
attn_impl: Literal["naive", "absorb"] = "absorb"

In [ ]:
# Config
class Config(PretrainedConfig):
    model_type = "mla_replica" # for later on: AutoConfig.register("ds_replica", Config)

    def __init__(
        self,
        vocab_size=6400,
        hidden_size=512,
        n_layers = 8,
        num_attention_heads=16,
        num_key_value_heads = 8,
        flash_attn = True,
        attention_bias = False,
        max_batch_size: int = 2,
        max_seq_len = 2048,
        intermediate_size = 2048,
        mlp_bias = False,
        dropout = 0.0,

        dtype: Literal["bf16", "fp8"] = "bf16",
        moe_intermediate_size = 256,
        
        # moe
        n_routed_experts: int = 64,
        n_shared_experts: int = 2,
        n_activated_experts: int = 6,
        n_expert_groups: int = 1,
        n_limited_groups: int = 1,
        score_func: Literal["softmax", "sigmoid"] = "softmax",
        route_scale: float = 1.,
        
        # mla
        q_lora_rank: int = 0,  
        kv_lora_rank: int = 128, 
        qk_nope_head_dim: int = 32,   # hidden_size/num_attention_heads, 512/16
        qk_rope_head_dim: int = 16,  # the dim size of rotary embedding per head
        v_head_dim: int = 32,  # usually same as qk_nope_head_dim
        
        # # yarn
        # original_seq_len: int = 4096,
        # rope_theta: float = 10000.0,
        # rope_factor: float = 40,
        # beta_fast: int = 32,
        # beta_slow: int = 1,
        # mscale: float = 1.,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.flash_attn = flash_attn
        self.attention_bias = attention_bias
        self.max_batch_size = max_batch_size
        self.max_seq_len = max_seq_len
        self.intermediate_size = intermediate_size
        self.mlp_bias = mlp_bias
        self.dropout = dropout
        self.dtype = dtype
        self.moe_intermediate_size = moe_intermediate_size
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts
        self.n_activated_experts = n_activated_experts
        self.n_expert_groups = n_expert_groups 
        self.n_limited_groups = n_limited_groups
        self.score_func = score_func
        self.route_scale = route_scale
        self.q_lora_rank = q_lora_rank
        self.kv_lora_rank = kv_lora_rank
        self.qk_nope_head_dim = qk_nope_head_dim
        self.qk_rope_head_dim = qk_rope_head_dim
        self.v_head_dim = v_head_dim
        # self.original_seq_len = original_seq_len
        # self.rope_theta = rope_theta
        # self.rope_factor = rope_factor
        # self.beta_fast = beta_fast
        # self.beta_slow = beta_slow
        # self.mscale = mscale
config = Config()
config.dropout

0.0

In [ ]:
# def repeat_kv(hidden_states, num_key_value_groups):
#     B, S, n_h, d_h = hidden_states.shape # at this moment, the k/v has been linearly projected in consideration of num_key_value_heads
#     if num_key_value_groups == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, :, None, :].expand(B, S, n_h, num_key_value_groups, d_h)
#     return hidden_states.reshape(B, S, n_h * num_key_value_groups, d_h)

# GPT architecture
class MLA(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.heads_dim = self.hidden_size // self.num_attention_heads # simpily using default ones instead of customized heads_dim
        # self.num_key_value_heads = config.num_key_value_heads
        # self.num_key_value_groups = self.num_attention_heads // self.num_key_value_heads
        # self.dropout_prob = config.dropout
        self.flash_attn = self.config.flash_attn
        self.k_cache, self.v_cache = None, None
        self.is_causal = True
        # self.dropout = nn.Dropout(self.dropout_prob) # simpily using the same value instead of distinguishing attention_dropout and residual_dropout

        self.q_lora_rank = config.q_lora_rank  # i.e. d_c'
        self.kv_lora_rank = config.kv_lora_rank # i.e. d_c
        self.qk_nope_head_dim = config.qk_nope_head_dim  # i.e. d_h
        self.qk_rope_head_dim = config.qk_rope_head_dim
        self.qk_head_dim = config.qk_nope_head_dim + config.qk_rope_head_dim # d_h + d^R_h
        self.v_head_dim = config.v_head_dim # d_h

        # down and up projection for mla
        self.wkv_a = nn.Linear(self.hidden_size, self.kv_lora_rank + self.qk_rope_head_dim) # down prj for hidden size, d->d_c+d^R_h; merge W^DKV and W^KR in one linear projection for easier computation
        self.kv_norm = RMSNorm(self.kv_lora_rank)
        self.wkv_b = nn.Linear(self.kv_lora_rank, self.num_attention_heads * (self.qk_nope_head_dim + self.v_head_dim))  # up prj for hidden size, d_c->2*n_h*d_h; merge W^UK and W^UV in one linear projection for easier computation

        if self.q_lora_rank == 0:
            self.wq = nn.Linear(self.hidden_size, self.num_attention_heads * self.qk_head_dim)
        else:
            self.wq_a = nn.Linear(self.hidden_size, self.q_lora_rank) # down prj for hidden size, d->d_c'
            self.q_norm = RMSNorm(self.q_lora_rank)
            self.wq_b = nn.Linear(self.q_lora_rank, self.num_attention_heads * self.qk_head_dim) # up prj for hidden size, d_c'->n_h*(d_h+d^R_h)

        self.wo = nn.Linear(self.num_attention_heads * self.v_head_dim, self.hidden_size) # n_h*d_h->d
        self.rotary_emb = RotaryEmbedding(self.qk_rope_head_dim)

        if attn_impl == 'naive':
            self.register_buffer('k_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.num_attention_heads, self.qk_head_dim), persistent=False)
            self.register_buffer('v_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.num_attention_heads, self.v_head_dim), persistent=False)

        else:
            self.register_buffer('kv_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.kv_lora_rank), persistent=False)
            self.register_buffer('pe_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.qk_rope_head_dim), persistent=False)


    def forward(self, hidden_states, mask=None, use_kv_cache=False, start_pos=0):
        # `start_pos` is the absolute position of the FIRST query token in this batch.
        # Prefill: start_pos=0, S = prompt_len. (S == end_pos)
        # Decode step: start_pos = past_len, S = 1 (just the newly generated token). (end_pos = start_pos + 1)
        # Training: use_kv_cache=False, start_pos=0 — math identical to before this refactor.
        # NOTE on q/k length: q's length is always S (current batch). k's length is S (no cache)
        # or end_pos (with cache, reads full prefix). The einsum letters `s` (for q) and `t` (for k)
        # below encode exactly this asymmetry.
        B, S, d = hidden_states.shape
        end_pos = start_pos + S

        kv = self.wkv_a(hidden_states) # (B, S, d_c + d^R_h)
        kv_nope, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        if self.q_lora_rank == 0:
            q = self.wq(hidden_states)
        else:
            q = self.wq_a(hidden_states) # (B, S, d_c')
            q = self.q_norm(q) # (B, S, d_c')
            q = self.wq_b(q) # (B, S, n_h*(d_h+d^R_h))
        q = q.view(B, S, self.num_attention_heads, self.qk_head_dim) # (B, S, n_h, d_h+d^R_h)
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)

        k_pe = k_pe.unsqueeze(2) # k_pe shape:(B, S, 1, d^R_h)
        q_pe, k_pe = self.rotary_emb(q_pe, k_pe, start_pos=start_pos)
        if attn_impl == 'naive':
            q = torch.cat([q_nope, q_pe], dim=-1) # (B, S, n_h, d_h+d^R_h)
            kv_nope = self.kv_norm(kv_nope)
            kv_nope = self.wkv_b(kv_nope) # (B, S, 2*n_h*d_h))
            kv_nope = kv_nope.view(B, S, self.num_attention_heads, self.qk_nope_head_dim + self.v_head_dim)
            k_nope, v_new = torch.split(kv_nope, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)

            k_new = torch.cat([k_nope, k_pe.expand(-1,-1,self.num_attention_heads,-1)], dim=-1) # (B, S, n_h, d_h+d^R_h)

            if use_kv_cache:
                self.k_cache[:B, start_pos:end_pos, :, :] = k_new
                self.v_cache[:B, start_pos:end_pos, :, :] = v_new
                k = self.k_cache[:B, :end_pos]  # (B, end_pos, n_h, qk_head_dim) -- full prefix incl. k_new
                v = self.v_cache[:B, :end_pos]  # (B, end_pos, n_h, v_head_dim)
            else:
                k, v = k_new, v_new              # (B, S, n_h, qk_head_dim) / (B, S, n_h, v_head_dim)

            scores = torch.einsum("bshd,bthd->bsht", q, k) / math.sqrt(self.qk_head_dim)
        else:
            # consider weights absortion to avoid calculating k distinctly, i.e. via changing multiply order i.e. A*(B*C) -> (A*B) * C to reduce computation cost
            wkv_b = self.wkv_b.weight # if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size) , (d_h*n_h, d_c)
            wkv_b = wkv_b.view(self.num_attention_heads, -1, self.kv_lora_rank) # (n_h, d_h, d_c)
            # q_{nope} = q_{nope} \times W^{UK}
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            kv_nope_new = self.kv_norm(kv_nope)
            k_pe_2d_new = k_pe.squeeze(2)

            if use_kv_cache:
                self.kv_cache[:B, start_pos:end_pos] = kv_nope_new
                self.pe_cache[:B, start_pos:end_pos] = k_pe_2d_new
                kv_nope = self.kv_cache[:B, :end_pos]  # (B, end_pos, kv_lora_rank)
                k_pe_2d = self.pe_cache[:B, :end_pos]  # (B, end_pos, qk_rope_head_dim)
            else:
                kv_nope = kv_nope_new                  # (B, S, kv_lora_rank)
                k_pe_2d = k_pe_2d_new                  # (B, S, qk_rope_head_dim)

            scores = (torch.einsum("bshc,btc->bsht", q_nope, kv_nope) +
                      torch.einsum("bshr,btr->bsht", q_pe, k_pe_2d)) / math.sqrt(self.qk_head_dim)
        if mask is not None:
            scores += mask.unsqueeze(1) 
        scores = scores.softmax(dim=-1)

        if attn_impl == 'naive':
            x = torch.einsum("bsht,bthd->bshd", scores, v)  # (B, S, n_h, d_h)
        else:
            # `kv_nope` is the cache slice during decode and the live latent during training.
            # Either way it carries the right K-side rows for the weighted sum.
            x = torch.einsum("bsht,btc->bshc", scores, kv_nope)
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])
        x = self.wo(x.flatten(2)) 
        return x

In [28]:
class FeedForward(nn.Module):
    # Note: why not using nn.Sequential() to implement SwiGLU? - cuz it's not linear pipeline but including parallel structure and a multiplicative operation
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.dropout = config.dropout
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=config.mlp_bias)

    def forward(self, hidden_states):
        down_proj = self.down_proj(F.silu(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))
        return down_proj


In [29]:
class DecoderLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.attention = MLA(config)
        self.ffn = FeedForward(config)
        self.input_layernorm = RMSNorm(self.hidden_size)
        self.post_attention_layernorm = RMSNorm(self.hidden_size)

    def forward(self, hidden_states, mask=None, use_kv_cache=False, start_pos=0):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.attention(hidden_states, mask=mask, use_kv_cache=use_kv_cache, start_pos=start_pos)
        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.ffn(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states

In [ ]:
class LLM(PreTrainedModel):
    config_class = Config  # for later on: AutoModelForCausalLM.register(Config, LLM)
    def __init__(self, config):
        super().__init__(config)
        self.vocat_size = self.config.vocab_size
        self.n_layers = self.config.n_layers
        self.dropout = nn.Dropout(self.config.dropout)
        self.token_embeddings = nn.Embedding(self.config.vocab_size, self.config.hidden_size)
        self.layers = torch.nn.ModuleList()
        for _ in range(self.n_layers):
            self.layers.append(DecoderLayer(config))
        self.layernorm = RMSNorm(self.config.hidden_size)
        self.output = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False) # each token generated's shape is (hidden_size, vocab_size)
        self.apply(self._init_weights)
        self.loss = None

        # TODO: Have no idea why it looks like this, looks so hacky - explained by GPT:
        # the loop over self.named_parameters() looks for tensor names ending with w3.weight (the MLP’s down-projection in a SwiGLU block)
        # or wo.weight (the attention output projection) and rescales them with a smaller std, 0.02 / sqrt(2 * n_layers), to match the RMSNorm-residual scaling used in LLaMA-style models.
        for pn, p in self.named_parameters():
            if pn.endswith('w3.weight') or pn.endswith('wo.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.config.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, input_ids, labels=None, use_kv_cache=False, start_pos=0):
        B, S = input_ids.shape
        end_pos = start_pos + S  # to track the entire seq length
        hidden_states = self.token_embeddings(input_ids)
        hidden_states = self.dropout(hidden_states)
        # Causal mask of shape (S_q, T_k). Query i is at absolute position (start_pos+i)
        # and may attend to key positions [0, start_pos+i]. T_k is end_pos when caching
        # (keys are the full prefix held in cache), else S (training: keys == queries).
        # Mask broadcasts to scores (B, S_q, n_h, T_k) via mask.unsqueeze(1) inside MLA.
        if S == 1 and start_pos > 0:
            # Decode step: the single new query at position start_pos can attend to
            # every cached key 0..start_pos — no positions to mask out.
            mask = None
        else:
            T_k = end_pos if use_kv_cache else S
            row = torch.arange(S, device=input_ids.device).unsqueeze(1)        # (S, 1)
            col = torch.arange(T_k, device=input_ids.device).unsqueeze(0)      # (1, T_k)
            mask = torch.where(col <= start_pos + row, 0.0, float('-inf'))
        for layer in self.layers:
            hidden_states = layer(hidden_states, mask=mask, use_kv_cache=use_kv_cache, start_pos=start_pos)
        hidden_states = self.layernorm(hidden_states)

        if labels is not None:
            logits = self.output(hidden_states)
            self.loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=0)
        else:
            # for inference
            logits = self.output(hidden_states[:, [-1], :])
            self.loss = None

        return CausalLMOutputWithPast(self.loss, logits) # meaning can call LLM().loss, LLM.logits directly

    @torch.inference_mode
    def generate(self, inputs, eos, max_new_tokens, temperature=0.7, top_k=None, stream=True, repetition_penalty=1.,
                 use_kv_cache=True):

        input_ids = inputs['input_ids']
        s = input_ids.shape[1]
        # start_pos = how many tokens are already represented in the KV cache.
        # First iteration: 0  -> forward the entire prompt (the "prefill" pass).
        # Subsequent iterations: forward only the single new token at position start_pos.
        # Without cache: start_pos stays 0 and we forward the full growing sequence each step
        # (quadratic-cost behavior, kept as a comparison baseline).
        start_pos = 0
        while input_ids.shape[1] < max_new_tokens - 1:
            tokens_in = input_ids[:, start_pos:] if use_kv_cache else input_ids
            inference_res = self.forward(tokens_in, labels=None,
                                         use_kv_cache=use_kv_cache, start_pos=start_pos)
            logits = inference_res.logits
            logits = logits[:, -1, :]

            # apply penaly for repetitive tokens
            for token in set(input_ids.tolist()[0]):
                logits[:, token] /= repetition_penalty

            if temperature == 0.0:
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1, generator=None)

            if idx_next == eos:
                break

            # IMPORTANT: advance start_pos BEFORE we cat the new token onto input_ids,
            # so the next iteration's `input_ids[:, start_pos:]` is just the new token.
            if use_kv_cache:
                start_pos = input_ids.shape[1]

            input_ids = torch.cat((input_ids, idx_next), dim=1)
            if stream:
                yield input_ids[:, s:]

        if not stream:
            yield input_ids[:, s:] 

In [52]:
tokenizer = AutoTokenizer.from_pretrained("../../sft/tokenizer")
tokenizer.bos_token = '<|im_start|>' # based on original data
tokenizer.eos_token = '<|im_end|>'
print("vocab size:", len(tokenizer), " bos:", tokenizer.bos_token_id, " eos:", tokenizer.eos_token_id)
tokenizer.add_special_tokens({'additional_special_tokens': ['<|im_start|>', '<|im_end|>']})
tokenizer.convert_tokens_to_ids(['<|im_start|>', '<|im_end|>'])

vocab size: 6400  bos: 1  eos: 1


[6400, 6401]

In [53]:
class LLMDataset(IterableDataset):
    """Pretraining dataset for our custom LLM (vocab=6400, minimind-style).

    Pads with token id 0 in both input_ids and labels. The model's CE loss uses
    ignore_index=0, so padded positions don't contribute to loss.

    Also pre-shifts: X = ids[:-1], Y = ids[1:]. This matches our custom LLM.forward
    which does NOT auto-shift internally (unlike HF Qwen2ForCausalLM).
    """
    def __init__(self, data_path, tokenizer, max_seq_len):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __iter__(self):
        return self.data_process()

    def data_process(self):
        with open(self.data_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = json.loads(line)
                text = line['text']
                input_ids = self.tokenizer.encode(text)
                text_len = len(input_ids)
                if text_len > self.max_seq_len:
                    input_ids = input_ids[:self.max_seq_len]
                else:
                    input_ids = input_ids + [0] * (self.max_seq_len - text_len)
                input_ids = np.array(input_ids)
                X = np.array(input_ids[:-1]).astype(np.int64)
                Y = np.array(input_ids[1:]).astype(np.int64)
                yield {
                    'input_ids': torch.from_numpy(X),
                    'labels':    torch.from_numpy(Y),
                }

In [54]:
dataset = LLMDataset("../../sft/dataset/pretrain_hq.jsonl", tokenizer, max_seq_len=512)

In [55]:
# Instantiate the custom MLA-based LLM from scratch. Uses the Config defined above
# (vocab=6400, hidden=512, 8 layers, MLA with kv_lora_rank=128). All params trainable.
config = Config(vocab_size=len(tokenizer))
model = LLM(config)
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

trainable params: 38,632,064


In [56]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model)

38632064

In [57]:
# Training from scratch on a small model — no gradient checkpointing, no bf16 needed.
# Bump max_steps after you confirm the loss decreases.
args = TrainingArguments(
    output_dir='./result/mla',
    num_train_epochs=1,
    do_train=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=5e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_steps=10,
    logging_steps=10,
    save_steps=10,
    report_to='none',
)

In [58]:
data_collator = DefaultDataCollator()
trainer = Trainer(model=model, args=args, train_dataset=dataset,
                  processing_class=tokenizer, data_collator=data_collator)
trainer.train(resume_from_checkpoint=False)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6401, 'bos_token_id': 6400}.
/Users/yingyao/miniconda3/envs/transformer-practice/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,7.747700


TrainOutput(global_step=10, training_loss=7.747718811035156, metrics={'train_runtime': 39.0829, 'train_samples_per_second': 0.512, 'train_steps_per_second': 0.256, 'total_flos': 2167921996800.0, 'train_loss': 7.747718811035156, 'epoch': 1.0})

In [59]:
# Save the whole custom LLM (weights + config) under ./model/mla. Small enough that
# `Trainer.save_model` writes the full model (~tens of MB, not GB like Qwen2).
trainer.save_model('./model/mla')
trainer.save_state()

In [60]:
# Reload the saved checkpoint via the HF Auto* registry. Requires registering our
# custom Config + LLM class first (only needs to happen once per kernel).
AutoConfig.register("mla_replica", Config)
AutoModelForCausalLM.register(Config, LLM)
reload_model = AutoModelForCausalLM.from_pretrained('./model/mla')

# Build a short prompt to test generation.
input_ids = [tokenizer.bos_token_id] + tokenizer.encode("1+1等于几?")
input_data = {'input_ids': torch.tensor(input_ids).unsqueeze(0), "labels": None}
input_data

{'input_ids': tensor([[6400,  731,   14,   20, 6239, 1919,   34]]),
 'labels': None}

In [61]:
# Use LLM.generate (our custom generator method) — yields the suffix tokens as it goes.
# With only 100 training steps the output will be gibberish; you mostly want to see that
# the cache + sparse-attention machinery doesn't crash.
for token in reload_model.generate(inputs=input_data, eos=tokenizer.eos_token_id,
                                   max_new_tokens=100, stream=False):
    print(tokenizer.decode(token[0]))

 Am�，觉得 Reimes告诉我跟，我apt震ers。


## Appendix

In [ ]:
# rmsnorm = RMSNorm(hidden_size=768)
# torch.manual_seed(1234)
# result = rmsnorm.forward(torch.randn(1, 1024, 768))
# # print(rmsnorm.weight)
# print(result)

tensor([[[-0.1105, -0.4911,  0.1613,  ...,  0.8285, -0.3369,  0.2555],
         [-0.3322,  2.6126,  0.8440,  ..., -1.1634, -0.8469,  2.3062],
         [-1.2211, -0.3163,  0.6839,  ..., -0.7464, -1.4905,  1.1278],
         ...,
         [ 2.8208, -0.5343,  1.3579,  ..., -1.1893, -0.1555, -0.4554],
         [ 0.9785,  1.2703, -1.8128,  ..., -0.1759, -0.0936, -0.4683],
         [ 0.4855,  0.7175,  0.9907,  ...,  0.7397, -0.5728,  0.2728]]],
       grad_fn=<MulBackward0>)


In [8]:
# import torch
# dim = 768
# max_seq_len=2048
# inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))  # 形状(dim/2)
# t = torch.arange(max_seq_len).float().unsqueeze(1)  # 形状(max_seq_len, 1)

# freqs = t @ inv_freq.unsqueeze(0)  #(max_seq_len, 1)*(1, dim/2) = (max_seq_len, dim/2)
# print(t)
# print(freqs.shape)

In [ ]:
# k = torch.randn((2, 3, 4, 5))
# q = torch.randn((2, 3, 4, 5))
# v = torch.randn((2, 3, 4, 5))
# mask = torch.randn((2, 3, 6, 6))
# mask = torch.triu(mask, diagonal=1)
# scores = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(5) 

In [24]:
# # encoded input looks like: 
# data_iter = iter(dataset)
# print(type(data_iter))
# sample = next(data_iter)
# input_ids = sample['input_ids']
# tokenizer.decode(input_ids)

<class 'generator'>


' <|im_start|>鉴别一组中文文章的风格和特点，例如官方、口语、文言等。需要提供样例文章才能准确鉴别不同的风格和特点。<|im_end|> <|im_start|>好的，现在帮我查一下今天的天气怎么样?今天的天气依据地区而异。请问你需要我帮你查询哪个地区的天气呢？<|im_end|> <|im_start|>打开闹钟功能，定一个明天早上七点的闹钟。好的，我已经帮您打开闹钟功能，闹钟将在明天早上七点准时响起。<|im_end|> <|im_start|>为以下场景写一句话描述：一个孤独的老人坐在公园长椅上看着远处。一位孤独的老人坐在公园长椅上凝视远方。<|im_end|> <|im_start|>非常感谢你的回答。请告诉我，这些数据是关于什么主题的？这些数据是关于不同年龄段的男女人口比例分布的。<|im_end|> <|im_start|>帮我想一个有趣的标题。这个挺有趣的："如何成为一名成功的魔术师" 调皮的标题往往会吸引读者的注意力。<|im_end|> <|im_start|>回答一个问题，地球的半径是多少？地球的平均半径约为6371公里，这是地球自赤道到两极的距离的平均值。<|im_end|> <|im_start|>识别文本中的语气，并将其分类为喜悦、悲伤、惊异等。\n文本：“今天是我的生日！”这个文本的语气是喜悦。<|im_end|><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [54]:
# input_ids = torch.randint(0, 10, (2, 512)) # min_int, max_int, (S, H)
# labels = torch.randint(0, 10, (2, 512))
# model(input_ids, labels).logits.shape

torch.Size([2, 512, 6400])

In [22]:
hidden_states = torch.randn(2, 100, 512)
attn_impl: Literal["naive", "absorb"] = "absorb"
mla = MLA(config=config)

# print(mla(hidden_states))
assert mla(hidden_states).shape == hidden_states.shape, 'attention output shape is wrong'
# print(mla.kv_cache)
assert mla.kv_cache.shape == torch.Size([2, config.max_seq_len, config.kv_lora_rank]), 'kv_cache shape is wrong'